In [1]:
import pandas as pd
import sys
sys.path.append("..")
from app.features import make_features
from app.backtest import walk_forward_splits

In [2]:
y = pd.read_parquet("../data/processed/hourly_jan2026.parquet")["trips"]
X = make_features(y).dropna()
y1 = y.loc[X.index]
folds = list(walk_forward_splits(X, 24*14, 24, 24, "expanding"))

In [3]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import mlflow
mlflow.set_experiment("demand-forecasting")

<Experiment: artifact_location='file:c:/Omid.h/Projects/demand-forecasting-service/notebooks/mlruns/1', creation_time=1788422101004, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788422101004, lifecycle_stage='active', name='demand-forecasting', tags={}, trace_location=None, workspace='default'>

In [4]:
params = {"num_leaves": 5, "min_child_samples": 7, 
          "n_estimators": 50, "random_state": 42}

with mlflow.start_run(run_name="lgbm_regularized"):
    mlflow.log_params(params)

    rows = []
    for tr, te in folds:
        model = LGBMRegressor(**params)
        
        model.fit(X.loc[tr], y1.loc[tr])
        y_pred = pd.Series(model.predict(X.loc[te]), index=te)
        y_true = y1.loc[te]
        rows.append({"MAE": mean_absolute_error(y_true, y_pred), "MAPE": mean_absolute_percentage_error(y_true, y_pred)})


    res = pd.DataFrame(rows)
    #res.agg(["mean", "median", "std"])    

    mlflow.log_metric("mape_median", res["MAPE"].median())
    mlflow.log_metric("mape_mean",   res["MAPE"].mean())
    mlflow.log_metric("mae_median",  res["MAE"].median())




[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000053 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 259
[LightGBM] [Info] Number of data points in the train set: 336, number of used features: 5
[LightGBM] [Info] Start training from score 5223.181548
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000029 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 275
[LightGBM] [Info] Number of data points in the train set: 360, number of used features: 5
[LightGBM] [Info] Start training from score 5253.247222
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000032 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 291
[LightGBM] [Info] Number of data points in the train set: 384, number of used features: 5
[LightGBM] [Info] Start training 

In [5]:
res.agg(["mean", "median", "std"])    

,MAE,MAPE
mean,1106.945989,0.495382
median,638.150455,0.156454
std,1000.400769,0.777676


In [8]:
eval_params = {"initial_train": 24*14, "horizon": 24, "step": 24, "scheme": "expanding"}

with mlflow.start_run(run_name="seasonal_naive"):
    mlflow.log_params({"model_type": "seasonal_naive", "seasonal_shift": 168, **eval_params})
    rows = []
    for tr, te in folds:
        y_pred = X.loc[te, "lag_168"]      # baseline = همون ستون، بدون fit
        y_true = y1.loc[te]

        mae = mean_absolute_error(y_true, y_pred)
        mape = mean_absolute_percentage_error(y_true, y_pred)

        
        rows.append({"MAE": mae, "MAPE": mape})
    res = pd.DataFrame(rows)
    mlflow.log_metric("mape_median", res["MAPE"].median())
    mlflow.log_metric("mape_mean",   res["MAPE"].mean())

In [ ]:
#mlflow.end_run()

In [16]:
with mlflow.start_run(run_name="lgbm_default"):
    model_default = LGBMRegressor(random_state=42)
    mlflow.log_params(model_default.get_params())
    mlflow.log_param("model_type", "lgbm_default")

    rows = []
    for tr, te in folds:          
        model_default.fit(X.loc[tr], y1.loc[tr])
        y_pred = pd.Series(model_default.predict(X.loc[te]), index=te)
        y_true = y1.loc[te]
        rows.append({"MAE": mean_absolute_error(y_true, y_pred), "MAPE": mean_absolute_percentage_error(y_true, y_pred)})
    
    
    res = pd.DataFrame(rows)
        #res.agg(["mean", "median", "std"])    
    
    mlflow.log_metric("mape_median", res["MAPE"].median())
    mlflow.log_metric("mape_mean",   res["MAPE"].mean())
    mlflow.log_metric("mae_median",  res["MAE"].median())


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000309 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 259
[LightGBM] [Info] Number of data points in the train set: 336, number of used features: 5
[LightGBM] [Info] Start training from score 5223.181548
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

In [ ]:
res

In [ ]:
pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)